In [ ]:
%pip install -q ucimlrepo


In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Model and simple comparison baseline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

# Evaluating predictions
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
adult = fetch_ucirepo(id=2)

# Combine the predictors and income label
raw_df = adult.data.features.join(adult.data.targets)

print("Dataset shape:", raw_df.shape)
display(raw_df.head())

Dataset shape: (48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [ ]:
def clean_adult(raw_df):
    df = raw_df.copy()

    # Remove extra spaces and mark missing values
    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = df[col].str.strip()

    df = df.replace('?', np.nan)

    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = df[col].fillna('Unknown')


    income_labels = df['income'].str.rstrip('.')
    df['above-50k-income'] = income_labels.map({
        '<=50K': 0,
        '>50K': 1
    })
    df['male-gender'] = df['sex'].map({'Female': 0, 'Male': 1})


    df['US-native'] = (df['native-country'] == 'United-States').astype(int)

    education_groups = {
        1: 'Elementary', 2: 'Elementary',
        3: 'Middle', 4: 'Middle',
        5: 'HS Partial', 6: 'HS Partial',
        7: 'HS Partial', 8: 'HS Partial',
        9: 'HS Grad', 10: 'HS Grad',
        11: 'Associate', 12: 'Associate',
        13: 'Bachelors',
        14: 'Masters', 15: 'Masters',
        16: 'Doctorate'
    }
    df['education'] = df['education-num'].map(education_groups)

    workclass_groups = {
        'Private': 'Private',
        'Federal-gov': 'Gov',
        'Local-gov': 'Gov',
        'State-gov': 'Gov',
        'Self-emp-not-inc': 'Self-Employed',
        'Self-emp-inc': 'Self-Employed',
        'Without-pay': 'Unemployed',
        'Never-worked': 'Unemployed',
        'Unknown': 'Unknown'
    }
    df['workclass'] = df['workclass'].map(workclass_groups).fillna('Unknown')

    return df[[
        'age', 'male-gender', 'US-native', 'race',
        'education', 'workclass', 'occupation',
        'hours-per-week', 'marital-status', 'relationship',
        'capital-gain', 'capital-loss', 'above-50k-income'
    ]]


final_df = clean_adult(raw_df)

print("Cleaned shape:", final_df.shape)
print("Missing values:", final_df.isna().sum().sum())
print("\nIncome counts:")
print(final_df['above-50k-income'].value_counts())
display(final_df.head())

Cleaned shape: (48842, 13)
Missing values: 0

Income counts:
above-50k-income
0    37155
1    11687
Name: count, dtype: int64


,age,male-gender,US-native,race,education,workclass,occupation,hours-per-week,marital-status,relationship,capital-gain,capital-loss,above-50k-income
0,39,1,1,White,Bachelors,Gov,Adm-clerical,40,Never-married,Not-in-family,2174,0,0
1,50,1,1,White,Bachelors,Self-Employed,Exec-managerial,13,Married-civ-spouse,Husband,0,0,0
2,38,1,1,White,HS Grad,Private,Handlers-cleaners,40,Divorced,Not-in-family,0,0,0
3,53,1,1,Black,HS Partial,Private,Handlers-cleaners,40,Married-civ-spouse,Husband,0,0,0
4,28,0,0,Black,Bachelors,Private,Prof-specialty,40,Married-civ-spouse,Wife,0,0,0


In [ ]:

numeric_features = [
    'age',
    'hours-per-week',
    'capital-gain',
    'capital-loss'
]

categorical_features = [
    'education',
    'workclass',
    'occupation',
    'marital-status',
    'relationship'
]

predictors = numeric_features + categorical_features

X = final_df[predictors]
y = final_df['above-50k-income']

groups = final_df['male-gender']

# Keep similar gender/income proportions across the splits
strata = groups.astype(str) + "_" + y.astype(str)

# Reserve 20% for the final fairness audit
X_dev, X_audit, y_dev, y_audit, group_dev, group_audit = (
    train_test_split(
        X, y, groups,
        test_size=0.20,
        random_state=42,
        stratify=strata
    )
)

# Split the remaining 80% into training and validation
dev_strata = group_dev.astype(str) + "_" + y_dev.astype(str)

X_train, X_val, y_train, y_val, group_train, group_val = (
    train_test_split(
        X_dev, y_dev, group_dev,
        test_size=0.25,
        random_state=42,
        stratify=dev_strata
    )
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Audit:", len(X_audit))
print("Number of predictors:", len(predictors))

Training: 29304
Validation: 9769
Audit: 9769
Number of predictors: 9


In [ ]:
# Prepare the numerical and categorical columns
preprocessor = ColumnTransformer([
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', OneHotEncoder(handle_unknown='ignore'),
     categorical_features)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [ ]:
val_predictions = model.predict(X_val)

# Simple baseline: always predict the most common income class
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)

print(f"Baseline accuracy: {baseline.score(X_val, y_val):.3f}")
print(f"Model accuracy: {accuracy_score(y_val, val_predictions):.3f}")
print(
    f"Balanced accuracy: "
    f"{balanced_accuracy_score(y_val, val_predictions):.3f}"
)

print("\nClassification report:")
print(classification_report(
    y_val,
    val_predictions,
    labels=[0, 1],
    target_names=['Income <=50K', 'Income >50K'],
    zero_division=0
))

Baseline accuracy: 0.761
Model accuracy: 0.851
Balanced accuracy: 0.762

Classification report:
              precision    recall  f1-score   support

Income <=50K       0.88      0.93      0.91      7432
 Income >50K       0.73      0.59      0.65      2337

    accuracy                           0.85      9769
   macro avg       0.81      0.76      0.78      9769
weighted avg       0.84      0.85      0.85      9769



In [ ]:
print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Validation labels:", len(y_val))
print("Audit rows:", len(X_audit))

print("Validation/audit overlap:",
      len(X_val.index.intersection(X_audit.index)))

print("Validation and audit are identical:",
      X_val.equals(X_audit))

Training rows: 29304
Validation rows: 9769
Validation labels: 9769
Audit rows: 9769
Validation/audit overlap: 0
Validation and audit are identical: False


In [ ]:
from sklearn.inspection import permutation_importance

# Measure importance using validation data only
importance = permutation_importance(
    model,
    X_val,
    y_val,
    scoring='balanced_accuracy',
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    'predictor': X_val.columns,
    'importance': importance.importances_mean,
    'shuffle_std': importance.importances_std
}).sort_values('importance', ascending=False)

display(importance_df.round(4))

,predictor,importance,shuffle_std
7,marital-status,0.1042,0.0049
4,education,0.0599,0.0031
6,occupation,0.0440,0.0023
2,capital-gain,0.0411,0.0016
8,relationship,0.0226,0.0030
1,hours-per-week,0.0184,0.0024
0,age,0.0153,0.0023
3,capital-loss,0.0098,0.0011
5,workclass,-0.0010,0.0013


In [ ]:
# Keep the original model and train a separate comparison model
reduced_categories = [
    col for col in categorical_features if col != 'workclass'
]
reduced_predictors = numeric_features + reduced_categories

reduced_preprocessor = ColumnTransformer([
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', OneHotEncoder(handle_unknown='ignore'),
     reduced_categories)
])

reduced_model = Pipeline([
    ('preprocessor', reduced_preprocessor),
    ('classifier', LogisticRegression(max_iter=2000))
])

reduced_model.fit(X_train[reduced_predictors], y_train)

reduced_predictions = reduced_model.predict(X_val[reduced_predictors])

from sklearn.metrics import recall_score

comparison = pd.DataFrame([
    {
        'Model': 'All 9 predictors',
        'Accuracy': accuracy_score(y_val, val_predictions),
        'Balanced accuracy': balanced_accuracy_score(y_val, val_predictions),
        'Recall >50K': recall_score(y_val, val_predictions)
    },
    {
        'Model': 'Without workclass',
        'Accuracy': accuracy_score(y_val, reduced_predictions),
        'Balanced accuracy': balanced_accuracy_score(y_val, reduced_predictions),
        'Recall >50K': recall_score(y_val, reduced_predictions)
    }
])

display(comparison.round(4))

,Model,Accuracy,Balanced accuracy,Recall >50K
0,All 9 predictors,0.8511,0.7619,0.5909
1,Without workclass,0.8513,0.7633,0.5948


In [ ]:
# Remove relationship
no_relationship_categories = [
    col for col in reduced_categories if col != 'relationship'
]
no_relationship_predictors = (
    numeric_features + no_relationship_categories
)

no_relationship_preprocessor = ColumnTransformer([
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', OneHotEncoder(handle_unknown='ignore'),
     no_relationship_categories)
])

no_relationship_model = Pipeline([
    ('preprocessor', no_relationship_preprocessor),
    ('classifier', LogisticRegression(max_iter=2000))
])

no_relationship_model.fit(
    X_train[no_relationship_predictors], y_train
)

no_relationship_predictions = no_relationship_model.predict(
    X_val[no_relationship_predictors]
)

# Compare all three versions
relationship_comparison = pd.concat([
    comparison,
    pd.DataFrame([{
        'Model': 'Without workclass or relationship',
        'Accuracy': accuracy_score(y_val, no_relationship_predictions),
        'Balanced accuracy': balanced_accuracy_score(
            y_val, no_relationship_predictions
        ),
        'Recall >50K': recall_score(y_val, no_relationship_predictions)
    }])
], ignore_index=True)

display(relationship_comparison.round(4))

,Model,Accuracy,Balanced accuracy,Recall >50K
0,All 9 predictors,0.8511,0.7619,0.5909
1,Without workclass,0.8513,0.7633,0.5948
2,Without workclass or relationship,0.8497,0.7601,0.5884


In [ ]:
# Check recall separately for females and males
validation_results = pd.DataFrame({
    'actual': y_val,
    'gender': group_val,
    'With relationship': reduced_predictions,
    'Without relationship': no_relationship_predictions
})

fairness_rows = []

for name in ['With relationship', 'Without relationship']:
    # Recall uses only people who actually earn >50K
    positives = validation_results[
        validation_results['actual'] == 1
    ]

    female = positives.loc[positives['gender'] == 0, name]
    male = positives.loc[positives['gender'] == 1, name]

    female_recall = female.mean()
    male_recall = male.mean()

    fairness_rows.append({
        'Model': name,
        'Female positives': len(female),
        'Male positives': len(male),
        'Female recall': female_recall,
        'Male recall': male_recall,
        'Gap (male - female)': male_recall - female_recall
    })

fairness_comparison = pd.DataFrame(fairness_rows)
display(fairness_comparison.round(4))

,Model,Female positives,Male positives,Female recall,Male recall,Gap (male - female)
0,With relationship,354,1983,0.4661,0.6178,0.1516
1,Without relationship,354,1983,0.4068,0.6208,0.2140


In [ ]:
# So we decided to remove the relationship variable because categories like husband and wife
# directly reveal gender. Removing it barely changed the overall accuracy
# (85.13% to 84.97%), but the female recall dropped from 46.61% to 40.68%,
# while male recall stayed around 62%. This increased the recall gap
# from about 15 to 21 percentage points on our validation data.
# So, removing a feature that reveals gender doesn't automatically make
# the model perform equally across groups. We still need to run the
# formal fairness tests on our held-out audit data.

In [ ]:
# Selected model: excludes workclass and relationship
final_model = no_relationship_model
final_predictors = no_relationship_predictors.copy()

print("Final predictors:")
for predictor in final_predictors:
    print("-", predictor)

print("\nModel: Logistic Regression")
print("Validation accuracy: 84.97%")
print("Validation recall gap: 21.40 percentage points")
print("Audit data has not been evaluated yet.")

Final predictors:
- age
- hours-per-week
- capital-gain
- capital-loss
- education
- occupation
- marital-status

Model: Logistic Regression
Validation accuracy: 84.97%
Validation recall gap: 21.40 percentage points
Audit data has not been evaluated yet.


In [ ]:
%pip install -q xgboost

In [21]:
from xgboost import XGBClassifier
from sklearn.base import clone

# Copy the preprocessing setup without its learned values
xgb_preprocessor = clone(no_relationship_preprocessor)

xgb_model = Pipeline([
    ('preprocessor', xgb_preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        n_jobs=2
    ))
])

xgb_model.fit(
    X_train[no_relationship_predictors],
    y_train
)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['age', 'hours-per-week',
                                                   'capital-gain',
                                                   'capital-loss']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['education', 'occupation',
                                                   'marital-status'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsa...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=2,
                               num_parallel_tree=None, ...))])

In [ ]:
# XGBoost predictions on validation data
xgb_predictions = xgb_model.predict(
    X_val[no_relationship_predictors]
)

# Identify actual higher earners within each group
female_positive = ((y_val == 1) & (group_val == 0)).to_numpy()
male_positive = ((y_val == 1) & (group_val == 1)).to_numpy()

results = []

for name, predictions in [
    ('Logistic Regression', no_relationship_predictions),
    ('XGBoost', xgb_predictions)
]:
    female_recall = predictions[female_positive].mean()
    male_recall = predictions[male_positive].mean()

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_val, predictions),
        'Balanced accuracy': balanced_accuracy_score(y_val, predictions),
        'Recall >50K': recall_score(y_val, predictions),
        'Female recall': female_recall,
        'Male recall': male_recall,
        'Gap (male - female)': male_recall - female_recall
    })

model_comparison = pd.DataFrame(results)
display(model_comparison.round(4))

,Model,Accuracy,Balanced accuracy,Recall >50K,Female recall,Male recall,Gap (male - female)
0,Logistic Regression,0.8497,0.7601,0.5884,0.4068,0.6208,0.2140
1,XGBoost,0.8613,0.7652,0.5811,0.4322,0.6077,0.1755


In [20]:
# So  when comparing logistic regression and XGBoost using the same seven
# predictors and training/validation splits. XGBoost improved accuracy
# from 84.97% to 86.13% and reduced the gender recall gap from 21.40
# to 17.55 percentage points. Female recall increased, but slightly worse at identifying males. It also identified slightly
# fewer higher earners overall, so it didn't improve everything.
# These are the results from the validation data.